# GWRF county-level validation figure

This notebook produces the final 2 × 3 validation figure for the Mongolian Plateau livestock spatialization model. The upper row presents the random 70/30 county-year holdout; the lower row presents county-grouped five-fold cross-validation. All panels show log-transformed county livestock density.

## 1. Packages and explicit file paths

Use Python 3.10 with the packages listed in requirements.txt. Input and output locations are explicitly defined below.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullLocator
from matplotlib import font_manager

print(font_manager.findfont("Arial", fallback_to_default=True))


VALIDATION_DIR = Path(r"E:\MPLD\SU\run\02_validation")

CODE_DIR = Path(r"E:\MPLD\code")
OUTPUT_DIR = Path(r"E:\MPLD\Figure")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_PREDICTIONS = VALIDATION_DIR / "random70_30_predictions.csv"
GROUPED_PREDICTIONS = VALIDATION_DIR / "county5fold_predictions.csv"
OUTPUT_STEM = OUTPUT_DIR / "Fig_GWRF_SU_validation"

## 2. Read validation predictions

In [ ]:
random_df = pd.read_csv(RANDOM_PREDICTIONS)
grouped_df = pd.read_csv(GROUPED_PREDICTIONS)

print(f"Random 70/30 records: {len(random_df):,}")
print(f"County-grouped five-fold records: {len(grouped_df):,}")

## 3. Metric and plotting functions

In [ ]:
def metric_text(data):
    observed = data["log_SU_density"].to_numpy(dtype=float)
    predicted = data["pred_log_SU_density"].to_numpy(dtype=float)
    residual = predicted - observed

    ss_res = np.sum(residual**2)
    ss_tot = np.sum((observed - observed.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot
    rmse = np.sqrt(np.mean(residual**2))
    mae = np.mean(np.abs(residual))

    return (
        rf"$R^2$ = {r2:.3f}" + "\n"
        + f"RMSE = {rmse:.3f}\n"
        + f"MAE = {mae:.3f}"
    )


def configure_figure_style():
    mpl.rcParams.update(
        {
            "font.family": "Arial",
            "font.size": 6.0,
            "axes.titlesize": 6.5,
            "axes.labelsize": 6.5,
            "xtick.labelsize": 5.8,
            "ytick.labelsize": 5.8,
            "axes.linewidth": 0.55,
            "xtick.major.width": 0.55,
            "ytick.major.width": 0.55,
            "xtick.major.size": 2.8,
            "ytick.major.size": 2.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
        }
    )

## 4. Define the six validation panels and common plotting scale

In [ ]:
panels = [
    ("a", "Mongolian Plateau", random_df),
    ("b", "Mongolia", random_df[random_df["country"] == "Mongolia"]),
    ("c", "Inner Mongolia", random_df[random_df["country"] == "Inner Mongolia"]),
    ("d", "Mongolian Plateau", grouped_df),
    ("e", "Mongolia", grouped_df[grouped_df["country"] == "Mongolia"]),
    ("f", "Inner Mongolia", grouped_df[grouped_df["country"] == "Inner Mongolia"]),
]

all_values = np.concatenate(
    [
        random_df["log_SU_density"].to_numpy(),
        random_df["pred_log_SU_density"].to_numpy(),
        grouped_df["log_SU_density"].to_numpy(),
        grouped_df["pred_log_SU_density"].to_numpy(),
    ]
)
finite = all_values[np.isfinite(all_values)]
margin = 0.04 * (finite.max() - finite.min())
lower = max(0.0, finite.min() - margin)
upper = finite.max() + margin

density_cmap = LinearSegmentedColormap.from_list(
    "density_blue",
    ["#EAF2F8", "#9CC6DE", "#3C86B6", "#123B5D"],
)

## 5. Draw the 160 mm double-column figure

In [ ]:
# ============================================================
# Figure style
# ============================================================
FONT_BASE = 8
FONT_LABEL = 8
FONT_ROW = 8
FONT_LEGEND = 8
FONT_TICK = 8

def configure_figure_style():
    plt.rcParams.update({
        # Font
        "font.family": "Arial",
        "font.sans-serif": ["Arial"],
        "font.size": FONT_BASE,
        "mathtext.fontset": "custom",
        "mathtext.rm": "Arial",
        "mathtext.it": "Arial:italic",
        "mathtext.bf": "Arial:bold",

        # Axes
        "axes.linewidth": 0.55,
        "axes.labelsize": FONT_LABEL,

        # Ticks
        "xtick.labelsize": FONT_TICK,
        "ytick.labelsize": FONT_TICK,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,

        # Legend
        "legend.fontsize": FONT_LEGEND,

        # Export
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        # Minus sign
        "axes.unicode_minus": False,
    })


configure_figure_style()


# ============================================================
# Figure layout
# ============================================================
figure_width_in = 160.0 / 25.4
figure_height_in = 100.0 / 25.4

fig, axes = plt.subplots(2, 3,
    figsize=(figure_width_in, figure_height_in),
    sharex=True, sharey=True)
fig.subplots_adjust(
    left=0.07,right=0.91,
    bottom=0.10,top=0.94,
    wspace=0.025,hspace=0.20)


# ============================================================
# Main panels
# ============================================================
hexbins = [] #六边形
for ax, (panel_label, region_name, data) in zip(axes.flat, panels):
    observed = data["log_SU_density"].to_numpy(dtype=float)
    predicted = data["pred_log_SU_density"].to_numpy(dtype=float)

    # Hexbin density
    hb = ax.hexbin(observed, predicted,gridsize=37,
        extent=(lower, upper, lower, upper),
        mincnt=1,linewidths=0,cmap=density_cmap)
    hexbins.append(hb)

    # Linear regression
    slope, intercept = np.polyfit(observed, predicted, 1)
    xline = np.array([lower, upper])

    ax.plot(xline, xline,color="#202020", linewidth=1.0,
        linestyle=(0, (4, 3)), zorder=3, label="1:1 line")
    ax.plot(xline, slope * xline + intercept,
        color="#B44B4B", linewidth=1, zorder=4, label="Linear fit")

    # Axes
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.55)
        spine.set_color("#202020")

    # Panel title
    ax.text(0.035, 0.965, f"({panel_label}) {region_name}",
        transform=ax.transAxes, ha="left", va="top", linespacing=1.50,
        bbox={"facecolor": "white", "edgecolor": "none",
            "alpha": 0.84, "pad": 0.8},
        zorder=7)

    # Legend
    ax.legend(loc="upper left",bbox_to_anchor=(0.03, 0.90),ncol=1,frameon=False,
        handlelength=1.55, handletextpad=0.35, labelspacing=0.25, borderaxespad=0)

    # R², RMSE, MAE
    ax.text(0.965, 0.055, metric_text(data),
        transform=ax.transAxes, ha="right", va="bottom",linespacing=1.25,
        bbox={"facecolor": "white", "edgecolor": "none",
              "alpha": 0.86, "pad": 2.0},
        zorder=7)


# ============================================================
# Shared hexbin normalization
# ============================================================
max_count = max(float(np.max(hb.get_array())) for hb in hexbins)
shared_norm = LogNorm(vmin=1, vmax=max_count)

for hb in hexbins:
    hb.set_norm(shared_norm)

# ============================================================
# Calculate actual subplot positions
# ============================================================
fig.canvas.draw()
top_row_pos = axes[0, 0].get_position()
bottom_row_pos = axes[1, 0].get_position()


# ============================================================
# Row headings
# ============================================================
fig.text(top_row_pos.x0, top_row_pos.y1 + 0.018,
    "Random split validation (70/30)",
    ha="left", va="bottom", fontsize=8, fontweight="bold")
fig.text(bottom_row_pos.x0, bottom_row_pos.y1 + 0.014,
    "County-grouped cross-validation (5-fold)",
    ha="left", va="bottom", fontsize=8, fontweight="bold")


# ============================================================
# Shared axis labels
# ============================================================
bottom_y = 0.03
fig.text(0.50, bottom_y,"Observed livestock density (log1p scale)",
    ha="center", va="center",fontsize=8)
fig.supylabel("Predicted livestock density (log1p scale)",
    x=0.040, y=0.50,fontsize=8)


# ============================================================
# Calculate actual subplot positions
# ============================================================
fig.canvas.draw()
top_left_pos = axes[0, 0].get_position()
bottom_left_pos = axes[1, 0].get_position()
top_right_pos = axes[0, -1].get_position()
bottom_right_pos = axes[1, -1].get_position()


# ============================================================
# Vertical colorbar
# ============================================================
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
full_bottom = bottom_right_pos.y0
full_top = top_right_pos.y1
full_height = full_top - full_bottom

# 短色带：占两排主图总高度的 38%
cbar_height = full_height * 0.38
cbar_y = full_bottom + (full_height - cbar_height) / 2

cax = fig.add_axes([
    top_right_pos.x1 + 0.018,   # 与第三列子图的距离
    cbar_y,
    0.012,                      # 色带宽度
    cbar_height
])

cbar = fig.colorbar(hexbins[0],cax=cax, orientation="vertical")
tick_values = [1, 10, 100]
tick_values = [v for v in tick_values if v <= max_count]

cbar.ax.yaxis.set_major_locator(FixedLocator(tick_values))
cbar.ax.yaxis.set_major_formatter(
    FixedFormatter([str(v) for v in tick_values]))
cbar.ax.yaxis.set_minor_locator(NullLocator())

cbar.set_label("Observation count",rotation=90, labelpad=5, fontsize=8)
cbar.outline.set_linewidth(0.45)
cbar.ax.tick_params(axis="y",
    which="major", labelsize=FONT_TICK,
    width=0.45, length=2.0, pad=2)

plt.show()

## 6. Export PNG, SVG and PDF

The figure is exported at its original 160 mm physical width. Automatic tight cropping is intentionally not used because it would alter the final physical dimensions.

In [ ]:
fig.savefig(OUTPUT_STEM.with_suffix(".png"), dpi=1200, facecolor="white")
fig.savefig(OUTPUT_STEM.with_suffix(".svg"))
fig.savefig(OUTPUT_STEM.with_suffix(".pdf"))

print(f"Saved: {OUTPUT_STEM.with_suffix('.png')}")
print(f"Saved: {OUTPUT_STEM.with_suffix('.svg')}")
print(f"Saved: {OUTPUT_STEM.with_suffix('.pdf')}")
plt.show()